In [ ]:
#@title 1. Setup &mdash; run once { display-mode: "form" }

#@markdown Tick this only if you want the files kept in your Google Drive. It asks
#@markdown you to sign in. Leave it off and everything goes to `/content/downloads`,
#@markdown which Colab wipes when the runtime disconnects.
use_drive = False  #@param {type:"boolean"}
drive_folder = "yt-downloads"  #@param {type:"string"}

#@markdown Leave blank and a key is generated for you and printed by cell 2.
api_key = ""  #@param {type:"string"}

import os, shutil, subprocess

REPO = "https://github.com/freelancermeer/ytmeer.git"
DIR  = "/content/ytmeer"

# ffmpeg merges the video+audio streams and reports the real quality, aria2c
# pulls each file over 16 connections, and node is the JS runtime yt-dlp needs
# for YouTube's "n" challenge. Colab already ships node.
!apt-get -qq install -y ffmpeg aria2 > /dev/null 2>&1
!pip -q install "yt-dlp>=2026.7.4" "gradio>=6,<7" "curl_cffi>=0.10,<0.16"

# The downloader itself - a pull if it is already here, a clone if not.
if os.path.isdir(os.path.join(DIR, ".git")):
    subprocess.run(["git", "-C", DIR, "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", REPO, DIR])

# Where the downloads go. Drive only if you asked for it.
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = f"/content/drive/MyDrive/{drive_folder.strip('/') or 'yt-downloads'}"
else:
    OUTDIR = "/content/downloads"
os.makedirs(OUTDIR, exist_ok=True)

# api.py and colab_app.py both read these, so cell 2 needs no arguments.
os.environ["YTDL_OUTDIR"] = OUTDIR
if api_key.strip():
    os.environ["YTDL_API_KEY"] = api_key.strip()

for tool in ("yt-dlp", "ffmpeg", "ffprobe", "aria2c", "node"):
    print(f"  {tool:8} {shutil.which(tool) or 'MISSING'}")
print(f"\n  downloads -> {OUTDIR}"
      f"{'  (your Drive - kept after the runtime ends)' if use_drive else '  (WIPED when the runtime disconnects)'}")
print("""
Now run cell 2. It prints one URL that carries all three of:

    the web UI          open it and click
    the REST API        GET/POST on <url>/api  - curl, requests, anything
    gradio_client       if you would rather use that

and the API key to go with it. Cookies are optional - public videos come down
without an account; add a cookies.txt in the UI only for private / members-only
videos, or if YouTube starts asking this runtime to confirm it is not a bot.
Anything that fails is reported with its reason, in the log and in the API.
""")

In [ ]:
#@title 2. Run &mdash; starts the UI and the API { display-mode: "form" }
import importlib, sys

sys.path.insert(0, "/content/ytmeer")
import api, colab_app
importlib.reload(api)           # pick up the folder and key cell 1 just set
importlib.reload(colab_app)

# share=True is what makes the URL reachable from outside Colab, which is what
# the REST API needs. Leave this cell running while you use it.
colab_app.launch(share=True)